# GRN controllability mini-notebook

> Conceptual anchors: Zañudo & Albert (2013, 2015), Mochizuki et al. (2013), Zañudo et al. (2017).

This notebook keeps the workflow intentionally short and computes, for a Boolean GRN:
- attractor seeds (stable-motif / succession-diagram view),
- minimal feedback vertex set (FVS),
- FVS-based control signatures for each attractor,
- observation duality metric: minimum number of FVS nodes needed to distinguish attractors from measurements.

In [ ]:
import itertools
import networkx as nx
import pandas as pd
import biobalm as balm
from biodivine_aeon import BooleanNetwork

pd.set_option("display.max_columns", None)

# Example GRN (replace with your own .bnet-style rules if needed).
RULE = """COUP,   !SP8&!PAX6&COUP | !FGF8 | EMX2
EMX2,   !SP8 | !FGF8 | EMX2
FGF8,   PAX6 | !COUP
PAX6,   !EMX2
SP8,    !EMX2
"""

In [ ]:
def get_attractor_states(rule: str) -> pd.DataFrame:
    sd = balm.SuccessionDiagram.from_rules(rule)
    sd.build()
    seed_dict = sd.expanded_attractor_seeds()

    rows = []
    for attr_id, states in sorted(seed_dict.items()):
        if not states:
            continue
        row = {"attractor": f"A{attr_id}"}
        row.update({k: int(v) for k, v in states[0].items()})
        rows.append(row)

    if not rows:
        return pd.DataFrame(columns=["attractor"])

    df = pd.DataFrame(rows)
    ordered_cols = ["attractor"] + sorted(c for c in df.columns if c != "attractor")
    return df[ordered_cols]


def regulatory_graph_from_rule(rule: str) -> nx.DiGraph:
    bn = BooleanNetwork.from_bnet(rule)
    graph = nx.DiGraph()

    for var in bn.variables():
        graph.add_node(bn.get_variable_name(var))

    for reg in bn.regulations():
        source = bn.get_variable_name(reg["source"])
        target = bn.get_variable_name(reg["target"])
        graph.add_edge(source, target)

    return graph


def minimal_fvs_sets(graph: nx.DiGraph):
    nodes = sorted(graph.nodes())
    for size in range(len(nodes) + 1):
        candidates = []
        for subset in itertools.combinations(nodes, size):
            reduced = graph.copy()
            reduced.remove_nodes_from(subset)
            if nx.is_directed_acyclic_graph(reduced):
                candidates.append(subset)
        if candidates:
            return candidates
    return []


def minimum_distinguishing_subsets(state_df: pd.DataFrame, candidate_nodes):
    if len(state_df) <= 1:
        return [tuple()]
    if not candidate_nodes:
        return []

    for size in range(1, len(candidate_nodes) + 1):
        good_subsets = []
        for subset in itertools.combinations(candidate_nodes, size):
            if not state_df.duplicated(subset=list(subset), keep=False).any():
                good_subsets.append(subset)
        if good_subsets:
            return good_subsets
    return []


def compute_controllability_metrics(rule: str):
    attractors = get_attractor_states(rule)
    graph = regulatory_graph_from_rule(rule)
    fvs_sets = minimal_fvs_sets(graph)
    primary_fvs = list(fvs_sets[0]) if fvs_sets else []

    if primary_fvs:
        fvs_signatures = attractors[["attractor"] + primary_fvs].copy()
        signatures_unique = not fvs_signatures.duplicated(subset=primary_fvs, keep=False).any()
    else:
        fvs_signatures = attractors[["attractor"]].copy()
        signatures_unique = len(attractors) <= 1

    control_rows = []
    for _, row in fvs_signatures.iterrows():
        assignment = {node: int(row[node]) for node in primary_fvs}
        control_rows.append(
            {
                "attractor": row["attractor"],
                "fvs_assignment": assignment,
                "n_fixed_nodes": len(primary_fvs),
                "fraction_fixed_nodes": len(primary_fvs) / graph.number_of_nodes() if graph.number_of_nodes() else 0.0,
            }
        )
    control_df = pd.DataFrame(control_rows)

    observation_subsets = minimum_distinguishing_subsets(fvs_signatures, primary_fvs)
    observation_df = pd.DataFrame(
        {"measurement_subset": [list(subset) for subset in observation_subsets]}
    )

    summary = pd.DataFrame(
        [
            {
                "n_nodes": graph.number_of_nodes(),
                "n_edges": graph.number_of_edges(),
                "n_attractors": len(attractors),
                "min_fvs_size": len(primary_fvs),
                "n_min_fvs_sets": len(fvs_sets),
                "signatures_unique_on_min_fvs": signatures_unique,
                "min_measurement_size_within_fvs": len(observation_subsets[0]) if observation_subsets else 0,
            }
        ]
    )

    fvs_sets_df = pd.DataFrame({"minimal_fvs_set": [list(s) for s in fvs_sets]})

    return {
        "graph": graph,
        "summary": summary,
        "attractors": attractors,
        "fvs_sets": fvs_sets_df,
        "fvs_signatures": fvs_signatures,
        "control_table": control_df,
        "observation_subsets": observation_df,
    }

In [ ]:
results = compute_controllability_metrics(RULE)

print("Regulatory edges:", sorted(results["graph"].edges()))
display(results["summary"])
display(results["fvs_sets"])
display(results["attractors"])
display(results["fvs_signatures"])
display(results["control_table"])
display(results["observation_subsets"])

## Reading the outputs

- `min_fvs_size`: smallest number of nodes that break all directed cycles in the interaction graph.
- `fvs_assignment` per attractor: control signature suggested by the FVS theorem (fix these nodes to attractor values).
- `signatures_unique_on_min_fvs` and `min_measurement_size_within_fvs`: observation-side duality (how many FVS nodes are sufficient to distinguish attractors).

For larger networks, the brute-force FVS search here can be replaced by a scalable approximation/solver-based routine.